# 07 — Gold Layer: Fact Trips

Consolidates all Silver collections into a unified **Fact Table** for the Star Schema.

Each fact record carries:
- All dimensional keys (date, zone, vehicle, vendor, rate_code, payment_type)
- Numeric measures (distance, duration, passengers, financials)
- **`silver_run_id`** — traces back to the Silver execution that produced this fact

This enables full Bronze → Silver → Gold lineage: given any Gold aggregate,
query `silver_run_id` → find the Silver run → its `bronze_run_id` → find the Bronze batch.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
from functools import reduce
import pyspark.sql.functions as F

print("Imports OK.")

In [ ]:
spark = get_spark("TLC_Gold_Fact")

## Flatten Silver Collections

In [ ]:
def read_and_flatten_silver(collection_name: str):
    """Read a Silver collection and project it to the fact table schema.

    silver_run_id — propagates the Silver execution ID into Gold,
    completing the full Bronze → Silver → Gold lineage chain.
    """
    df = read_mongo(spark, settings.MONGO_DB_SILVER, collection_name)
    return df.select(
        # ── Dimensional keys ────────────────────────────────────────────────
        F.date_format(F.col("datetimes.pickup"),  "yyyyMMdd").cast("int").alias("pickup_date_id"),
        F.date_format(F.col("datetimes.dropoff"), "yyyyMMdd").cast("int").alias("dropoff_date_id"),
        F.hour(F.col("datetimes.pickup")).alias("pickup_hour"),
        F.col("locations.pickup.zone_id").alias("pickup_zone_id"),
        F.col("locations.dropoff.zone_id").alias("dropoff_zone_id"),
        F.coalesce(F.col("vendor_id"), F.lit(-1)).alias("vendor_id"),
        F.coalesce(F.col("metrics.rate_code_id"), F.lit(-1)).alias("rate_code_id"),
        F.coalesce(F.col("financials.payment_type"), F.lit("Unknown")).alias("payment_type_id"),
        F.col("_meta.vehicle_type").alias("vehicle_id"),

        # ── Measures ────────────────────────────────────────────────────────
        F.coalesce(F.col("metrics.distance_miles"),    F.lit(0.0)).alias("trip_distance"),
        F.coalesce(F.col("datetimes.duration_minutes"), F.lit(0.0)).alias("duration_minutes"),
        F.coalesce(F.col("metrics.passenger_count"),   F.lit(0)).alias("passenger_count"),
        F.coalesce(F.col("financials.fare_amount"),    F.lit(0.0)).alias("fare_amount"),
        F.coalesce(F.col("financials.tip_amount"),     F.lit(0.0)).alias("tip_amount"),
        F.coalesce(F.col("financials.tolls_amount"),   F.lit(0.0)).alias("tolls_amount"),
        F.coalesce(F.col("financials.congestion_surcharge"), F.lit(0.0)).alias("congestion_surcharge"),
        F.coalesce(F.col("financials.cbd_congestion_fee"),   F.lit(0.0)).alias("cbd_congestion_fee"),
        F.coalesce(F.col("financials.total_amount"),   F.lit(0.0)).alias("total_amount"),

        # ── Lineage: Silver → Gold traceability ─────────────────────────────
        F.col("_meta.run_id").alias("silver_run_id"),
    )

In [ ]:
collections = ["trips_yellow", "trips_green", "trips_fhv", "trips_hvfhv"]
dfs = []

for coll in collections:
    try:
        df = read_and_flatten_silver(coll)
        n = df.count()
        dfs.append(df)
        print(f"  ✓ {coll}: {n:,} records")
    except Exception as e:
        print(f"  ✗ {coll}: {e}")

print(f"\nCollections loaded: {len(dfs)}")

## Union & Write to Gold

In [ ]:
if dfs:
    fact_trips_df = reduce(
        lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
        dfs,
    )
    n_total = fact_trips_df.count()
    write_mongo(fact_trips_df, settings.MONGO_DB_GOLD, "fact_trips", mode="overwrite")
    print(f"\nfact_trips written to tlc_gold: {n_total:,} total records")
    print("Silver run_id lineage is preserved in each fact row.")
else:
    print("No Silver collections found. Run notebooks 02-05 first.")

## Lineage Demo

Shows a sample of `silver_run_id` values per vehicle type — proof of full traceability.

In [ ]:
if dfs:
    fact_trips_df.groupBy("vehicle_id", "silver_run_id").count() \
        .orderBy("vehicle_id") \
        .show(20, truncate=False)